# 02 — Entrenamiento LoRA (Persona B)

Ajuste fino de **Qwen/Qwen2.5-Coder-1.5B-Instruct** con LoRA para Text-to-SQL (MySQL 8.4) usando TRL `SFTTrainer`.

- **Formato prompt–completion conversacional**: `prompt = build_messages(esquema, pregunta)` (el mismo `prompt.py` que usa `solucion.py`) y `completion = [{"role": "assistant", "content": sql}]`. Así la pérdida se calcula **solo sobre el SQL** (`completion_only_loss=True`).
- **Dos configuraciones, un solo factor (`target_modules`)**: A = atención; B = atención + MLP.
- El entrenamiento corre en `scripts/entrenar.py`, lanzado como subproceso para que cada config libere la GPU al terminar. Este notebook mide los datos, lanza las dos corridas y compara.

Salidas: `checkpoints/config_A|B/` (adapters, fuera de git), `resultados/entrenamiento_config_A|B.json`, `resultados/curvas_loss_A|B.png`, `resultados/curvas_loss_AB.png`, y el mejor adapter copiado a `adapter/`.

## 1. Parámetros

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATOS = ROOT / "datos" / "entrenamiento.jsonl"   # pares finales de A
EVAL = ROOT / "datos" / "eval.jsonl"             # opcional; si no existe: campo `particion`/`split` o 5 % por esquema
RESULTADOS = ROOT / "resultados"
LIMITE = None        # p. ej. 200 para la prueba temprana
EPOCHS = 2
BATCH, ACCUM = 1, 16 # batch efectivo 16; 2 x 8 roza los 8 GB de la RTX 5050 con esquemas de ~2k tokens
EVAL_STEPS = 50
FP16 = False         # True solo si se entrena en una T4 (sin bf16)
TOKENS_POR_SEG = 410 # medido en la RTX 5050 (config B, prueba de estrés); solo para estimar horas

## 2. Entorno

In [ ]:
import json, subprocess, sys, os, time
import torch, transformers, peft, trl, datasets

print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| datasets", datasets.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU",
      f"| {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB" if torch.cuda.is_available() else "",
      "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "scripts"))
from prompt import SYSTEM, build_messages
import entrenar
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
cfg

## 3. Datos y elección de `max_length`

`max_length` = p99 de la longitud en tokens (prompt + SQL con la plantilla de chat de Qwen), redondeado a múltiplo de 128. Los pares más largos se descartan (no se truncan: truncar cortaría justo el SQL, que es lo único con pérdida).

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer

registros = entrenar.leer_jsonl(DATOS)
df = pd.DataFrame(registros)
print(f"{len(df)} pares, {df['db_id'].nunique()} esquemas")
display(pd.crosstab(df["fuente"], df["nivel"], margins=True))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg["modelo_base"])
df["tokens"] = [entrenar.longitud_tokens(tokenizer, r) for r in registros]
df["tokens_sql"] = [len(tokenizer(r["sql"])["input_ids"]) for r in registros]
pct = df["tokens"].quantile([0.5, 0.9, 0.95, 0.99]).round()
print(pct.to_string(), "\nmax", df["tokens"].max())
MAX_LENGTH = int(np.ceil(pct[0.99] / 128) * 128)
print(f"\nMAX_LENGTH = {MAX_LENGTH}  ->  se descartan {(df['tokens'] > MAX_LENGTH).sum()} pares")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(df["tokens"], bins=50, color="#2a6fdb")
axes[0].axvline(MAX_LENGTH, color="#e0782d", ls="--", label=f"max_length={MAX_LENGTH}")
axes[0].set_title("Tokens por ejemplo (prompt + SQL)"); axes[0].legend()
axes[1].hist(df["tokens_sql"], bins=50, color="#2a6fdb")
axes[1].set_title("Tokens del SQL (lo que se aprende)")
plt.tight_layout(); plt.show()

In [ ]:
train, ev = entrenar.dividir(registros, EVAL, 0.05, cfg["semilla"])
print(f"train={len(train)}  eval={len(ev)}")
comunes = {r["db_id"] for r in train} & {r["db_id"] for r in ev}
print(f"esquemas en train={len({r['db_id'] for r in train})}, en eval={len({r['db_id'] for r in ev})}, compartidos={len(comunes)}")
print("(el eval es solo para la curva de loss; la validación real es la de C, con esquemas que NO están aquí)")

ej = entrenar.a_prompt_completion(train[0])
print("\n--- ejemplo de prompt tal como lo ve el modelo ---")
print(tokenizer.apply_chat_template(ej["prompt"] + ej["completion"], tokenize=False)[:1500])

In [ ]:
tok_train = sum(min(entrenar.longitud_tokens(tokenizer, r), MAX_LENGTH) for r in (train[:LIMITE] if LIMITE else train))
horas = tok_train * EPOCHS / TOKENS_POR_SEG / 3600
print(f"{tok_train:,} tokens por época  ->  ~{horas:.1f} h por config a {TOKENS_POR_SEG} tok/s ({EPOCHS} épocas)")
print("Si pasa de ~5 h por config: entrenar en Kaggle/Colab T4 (FP16=True, BATCH, ACCUM = 2, 8).")

## 4. Hipótesis (registradas en `BITACORA.md` **antes** de entrenar)

| Config | `target_modules` | Hipótesis |
|---|---|---|
| **A** | `q_proj, k_proj, v_proj, o_proj` | Aprende el formato y el dialecto MySQL, pero mejora poco en preguntas difíciles. |
| **B** | A + `gate_proj, up_proj, down_proj` | Con ~4.2× más parámetros entrenables (18.5 M vs 4.4 M, medido en la prueba temprana), mejora más en preguntas medias y difíciles. |

Hiperparámetros comunes: `r=16`, `lora_alpha=32`, `lora_dropout=0.05`, `lr=2e-4` cosine con warmup 3 %, 2 épocas, batch efectivo 16, gradient checkpointing, bf16 (RTX) / fp16 (T4), semilla 2026.

In [ ]:
def entrenar_config(nombre: str) -> dict:
    cmd = [sys.executable, str(ROOT / "scripts" / "entrenar.py"), "--config", nombre,
           "--datos", str(DATOS), "--eval", str(EVAL), "--resultados", str(RESULTADOS),
           "--max-length", str(MAX_LENGTH), "--epochs", str(EPOCHS),
           "--batch", str(BATCH), "--accum", str(ACCUM), "--eval-steps", str(EVAL_STEPS)]
    if LIMITE:
        cmd += ["--limite", str(LIMITE), "--salida", str(ROOT / "checkpoints" / f"prueba_{nombre}")]
    if FP16:
        cmd.append("--fp16")
    env = {**os.environ, "PYTHONIOENCODING": "utf-8"}
    proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    for linea in proc.stdout:
        # Hide progress bars; keep losses, warnings and errors.
        if "it/s]" not in linea and "s/it]" not in linea and "examples/s]" not in linea:
            print(linea, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"config {nombre} falló (código {proc.returncode})")
    etiqueta = nombre if not LIMITE else f"{nombre}_prueba"
    return json.loads((RESULTADOS / f"entrenamiento_config_{etiqueta}.json").read_text(encoding="utf-8"))

## 5. Config A — solo atención

In [ ]:
res_A = entrenar_config("A")
{k: res_A[k] for k in ["parametros_entrenables", "porcentaje_entrenable", "tiempo_legible", "loss_train_final", "loss_eval_final", "memoria_pico_gb"]}

In [ ]:
from IPython.display import Image
Image(filename=str(RESULTADOS / f"curvas_loss_{'A' if not LIMITE else 'A_prueba'}.png"))

## 6. Config B — atención + MLP

In [ ]:
res_B = entrenar_config("B")
{k: res_B[k] for k in ["parametros_entrenables", "porcentaje_entrenable", "tiempo_legible", "loss_train_final", "loss_eval_final", "memoria_pico_gb"]}

In [ ]:
Image(filename=str(RESULTADOS / f"curvas_loss_{'B' if not LIMITE else 'B_prueba'}.png"))

## 7. Comparación A vs B

In [ ]:
tabla = pd.DataFrame([res_A, res_B]).set_index("config").astype(object)[
    ["parametros_entrenables", "porcentaje_entrenable", "tiempo_legible", "pasos",
     "loss_train_final", "loss_eval_final", "tokens_por_segundo", "memoria_pico_gb", "precision", "gpu"]]
tabla.loc["B/A"] = ["%.2fx" % (res_B["parametros_entrenables"] / res_A["parametros_entrenables"]), "",
                    "%.2fx" % (res_B["tiempo_segundos"] / res_A["tiempo_segundos"]), "", "", "", "", "", "", ""]
tabla

In [ ]:
sufijo = "" if not LIMITE else "_prueba"
fig, ax = plt.subplots(figsize=(8, 4))
for nombre, color in (("A", "#2a6fdb"), ("B", "#e0782d")):
    hist = json.loads((RESULTADOS / f"log_config_{nombre}{sufijo}.json").read_text())
    tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    ev_ = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
    ax.plot(*zip(*tr), color=color, alpha=0.45, lw=1, label=f"{nombre} train")
    ax.plot(*zip(*ev_), color=color, lw=2, marker="o", ms=4, label=f"{nombre} eval")
ax.set_xlabel("paso de optimización"); ax.set_ylabel("loss (solo SQL)")
ax.set_title("Config A (atención) vs Config B (atención + MLP)"); ax.grid(alpha=.3); ax.legend()
fig.tight_layout(); fig.savefig(RESULTADOS / f"curvas_loss_AB{sufijo}.png", dpi=150); plt.show()

## 8. Chequeo rápido de generación

Unos ejemplos del eval con cada adapter usando **el mismo `solucion.py`** (variable `SOLUCION_CFG`). Es solo para ver que la salida es SQL limpio; la métrica real (EX ponderada sobre la validación propia) la calcula C con `scripts/evaluador.py`.

In [ ]:
muestra = ev[:5]
(RESULTADOS / "_muestra_tmp.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in muestra), encoding="utf-8")
script = '''
import json, sys, time
sys.path.insert(0, sys.argv[1])
import solucion
for l in open(sys.argv[2], encoding="utf-8"):
    r = json.loads(l); t = time.time()
    sql = solucion.generar_sql(r["esquema"], r["pregunta"])
    print(f"[{time.time()-t:4.1f}s] {r['pregunta']}\\n  pred: {sql}\\n  ref:  {r['sql']}")
'''
for nombre in ("A", "B"):
    carpeta = f"checkpoints/{'config_' if not LIMITE else 'prueba_'}{nombre}"
    env = {**os.environ, "PYTHONIOENCODING": "utf-8", "SOLUCION_CFG": json.dumps({"adapter": carpeta, "usar_adapter": True})}
    out = subprocess.run([sys.executable, "-c", script, str(ROOT), str(RESULTADOS / "_muestra_tmp.jsonl")],
                         cwd=ROOT, env=env, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(f"===== config {nombre} =====\n{out.stdout}{out.stderr[-2000:] if out.returncode else ''}")
(RESULTADOS / "_muestra_tmp.jsonl").unlink()

## 9. Adapter final

C evalúa A y B (y la autocorrección) en la validación propia. **La config con mayor EX ponderada** se copia a `adapter/` (que sí va a git, ~20–80 MB). Cambiar `MEJOR` según el resultado de C.

In [ ]:
import shutil

MEJOR = None   # "A" o "B" según la EX de C

if MEJOR and not LIMITE:
    origen = ROOT / "checkpoints" / f"config_{MEJOR}"
    destino = ROOT / "adapter"
    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir()
    for f in ("adapter_config.json", "adapter_model.safetensors"):
        shutil.copy2(origen / f, destino / f)
    print({f.name: f"{f.stat().st_size / 2**20:.1f} MB" for f in destino.iterdir()})
else:
    print("Sin copiar: falta elegir MEJOR (o es la prueba temprana).")